# 04｜O2O 加算收益与持有段计数分析

本 Notebook 的输入是 **03 Notebook 已生成的含持有期精简八列表**，不重新生成八列。

- 每行是实际执行日：前一实际交易日收盘形成，当前执行日开盘可执行；
- 收益使用执行日开盘到下一实际交易日开盘的 O2O；
- 曲线使用 `1 + cumsum(持仓 × O2O)` 的加算口径，不做复利；
- 最新行如果还没有下一实际交易日开盘价，保留信号和审计行，但不进入收益评价，不填 0；
- 本 Notebook 负责收益、净值、风险和持有段/段计数；逐年拆解及 2026 年原因分析由 05 完成。

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'src' / 'generate_compact_output.py').is_file()
)
SRC_ROOT = PACKAGE_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

SPOT_TEXT = os.environ.get(
    'COMPANY_SPOT_PATH',
    '/home/hzy/cta/IC数据更新_最终固化版/现货最终版/CSI500_SPOT_md_eod_raw_最终版.parquet',
).strip()
if not SPOT_TEXT or not Path(SPOT_TEXT).expanduser().is_absolute():
    raise RuntimeError('请设置 COMPANY_SPOT_PATH 为本地米筐现货的绝对路径。')
SPOT_PATH = Path(SPOT_TEXT).expanduser().resolve()
HOLDING_PATH = Path(os.environ.get(
    'HOLDING_EIGHT_PATH',
    str(PACKAGE_ROOT / 'runtime_outputs_holding_period' / '含持有期八列表.csv'),
)).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get(
    'ANALYSIS_04_OUTPUT_DIR',
    str(PACKAGE_ROOT / 'runtime_outputs_04_returns'),
)).expanduser().resolve()
if not HOLDING_PATH.is_absolute() or not OUTPUT_DIR.is_absolute():
    raise RuntimeError('04 的八列表路径和输出目录都必须是绝对路径。')

from reproduce_remote_o2o import run_stage_04

print('冻结包目录：', PACKAGE_ROOT)
print('本地现货：', SPOT_PATH)
print('03 八列表：', HOLDING_PATH)
print('04 输出：', OUTPUT_DIR)

冻结包目录： /Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包
本地现货： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824_1750/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet
03 八列表： /Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/runtime_outputs_holding_period/含持有期八列表.csv
04 输出： /Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/runtime_outputs_04_returns


## 1. 运行 O2O 加算收益与持有段分析

In [2]:
metadata = run_stage_04(SPOT_PATH, HOLDING_PATH, OUTPUT_DIR)
risk = pd.read_csv(OUTPUT_DIR / 'O2O加算风险指标.csv', encoding='utf-8-sig')
segment_counts = pd.read_csv(OUTPUT_DIR / '持有段计数_按系列.csv', encoding='utf-8-sig')
daily = pd.read_csv(OUTPUT_DIR / 'O2O加算逐日收益与状态.csv', encoding='utf-8-sig', parse_dates=['实际执行日', '推定形成日'])
display(risk)
display(segment_counts)
display(daily.tail(10))
print('生成文件数：', len(metadata['generated_files']))

/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:790: UserWarning: Glyph 21021 (\N{CJK UNIFIED IDEOGRAPH-521D}) missing from current font.
  fig.tight_layout()
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:790: UserWarning: Glyph 22987 (\N{CJK UNIFIED IDEOGRAPH-59CB}) missing from current font.
  fig.tight_layout()
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:790: UserWarning: Glyph 19977 (\N{CJK UNIFIED IDEOGRAPH-4E09}) missing from current font.
  fig.tight_layout()
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:790: UserWarning: Glyph 29366 (\N{CJK UNIFIED IDEOGRAPH-72B6}) missing from current font.
  fig.tight_layout()
/Users/hzy/Desktop/0817合并查看/20260824_1805_冻结中证500输出包/src/reproduce_remote_o2o.py:790: UserWarning: Glyph 24577 (\N{CJK UNIFIED IDEOGRAPH-6001}) missing from current font.
  fig.tight_layout()
/Users/hzy/Desktop/0817合并查看/20260824_180

,series,start_date,end_date,observations,final_nav,total_return_pct,annualized_additive_return_pct,annualized_volatility_pct,sharpe,max_drawdown_pct,active_days,winning_directional_days,directional_win_rate_pct,daily_win_rate_pct
0,Raw 1545,2018-01-03,2026-08-21,2095,2.437619,143.761913,17.292602,15.052685,1.148805,-11.699580,639,353,55.242567,16.849642
1,Adj 1545 (+4 Reversal),2018-01-03,2026-08-21,2095,3.831570,283.157004,34.059936,17.784170,1.915183,-10.949237,1088,608,55.882353,29.021480
2,CSI500 Buy & Hold,2018-01-03,2026-08-21,2095,1.455022,45.502225,5.473299,24.034090,0.227731,-45.905210,2095,1083,51.694511,51.694511


,series,state,segments,holding_days,mean_holding_days
0,Adj 1545,-1,132,434,3.287879
1,Adj 1545,1,124,656,5.290323
2,Raw 1545,-1,38,233,6.131579
3,Raw 1545,1,35,406,11.600000


,实际执行日,三状态,+1反转,-1反转,0转-1,0转+1,大涨,大跌,调整后三状态,调整原因,...,原始策略日收益,调整策略日收益,指数日收益,O2O可评价,原始净值,调整净值,指数净值,原始超额净值,调整超额净值,调整相对原始净值
2087,2026-08-12,0,0,0,1,0,0,0,-1,0转-1持有,...,0.0,-0.014892,0.014892,True,2.437619,3.823704,1.483837,1.953782,3.339867,2.386085
2088,2026-08-13,0,0,0,1,0,0,0,-1,0转-1持有,...,-0.0,0.013240,-0.013240,True,2.437619,3.836945,1.470597,1.967022,3.366348,2.399325
2089,2026-08-14,0,0,0,1,0,0,0,-1,0转-1持有,...,0.0,-0.002281,0.002281,True,2.437619,3.834664,1.472878,1.964742,3.361786,2.397045
2090,2026-08-17,0,0,0,1,0,0,0,-1,0转-1持有,...,0.0,-0.021351,0.021351,True,2.437619,3.813313,1.494228,1.943391,3.319085,2.375694
2091,2026-08-18,0,0,0,1,0,0,0,-1,0转-1持有,...,-0.0,0.018257,-0.018257,True,2.437619,3.831570,1.475971,1.961648,3.355599,2.393951
2092,2026-08-19,0,0,0,0,0,0,0,0,基础0,...,-0.0,-0.000000,-0.020530,True,2.437619,3.831570,1.455441,1.982178,3.376129,2.393951
2093,2026-08-20,0,0,0,0,0,0,0,0,基础0,...,-0.0,-0.000000,-0.006665,True,2.437619,3.831570,1.448777,1.988842,3.382793,2.393951
2094,2026-08-21,0,0,0,0,0,0,0,0,基础0,...,0.0,0.000000,0.006246,True,2.437619,3.831570,1.455022,1.982597,3.376548,2.393951
2095,2026-08-24,0,0,0,1,0,0,1,-1,0转-1持有,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN
2096,2026-08-25,0,0,0,1,0,0,0,-1,0转-1持有,...,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN


生成文件数： 25


## 2. 日期、收益和最新行检查

这里明确检查：实际执行日行的收益只取当前执行日开盘到下一实际交易日开盘；最新行没有下一开盘时只能是未评价状态，不能被补成占位收益。

In [3]:
if not daily['形成日早于执行日'].all():
    raise AssertionError('发现形成日不早于执行日的行')
valid = daily['O2O可评价']
expected_o2o = daily.loc[valid, '下一交易日开盘'] / daily.loc[valid, '执行日开盘'] - 1.0
actual_o2o = daily.loc[valid, '执行日O2O']
if (expected_o2o - actual_o2o).abs().max() > 1e-12:
    raise AssertionError('O2O 不是执行日开盘到下一实际交易日开盘')
if daily['实际执行日'].duplicated().any():
    raise AssertionError('实际执行日重复')
latest = daily.iloc[-1]
print('最新形成日：', latest['推定形成日'])
print('最新执行日：', latest['实际执行日'])
print('最新行有下一开盘价：', bool(latest['O2O可评价']))
print('可评价行数：', int(valid.sum()), '/', len(daily))
print('检查通过：没有用占位值补齐最新收益。')

最新形成日： 2026-08-24 00:00:00
最新执行日： 2026-08-25 00:00:00
最新行有下一开盘价： False
可评价行数： 2095 / 2097
检查通过：没有用占位值补齐最新收益。


## 3. 输出位置

04 会输出 `O2O加算逐日收益与状态.csv`、`持有段明细.csv`、`持有段计数_按系列.csv`、风险指标和四张收益曲线。05 Notebook 只读取这些 04 结果做逐年与 2026 分析。

另外，04 会单独输出只加入 `0→-1`、`0→+1` 的含持有期场景：`持有段明细_仅零段反转.csv`（每一段的开始执行日、结束执行日、持有交易日数、来源）和 `持有段天数分布_仅零段反转.csv`。该场景不加入 `-1→0`、`+1→0`。

## 4. 只加入两个零段反转后的持仓段日期

下面的场景从基础三状态出发，只在基础状态为 0 时叠加 03 生成的连续 `0转-1`、`0转+1` 持有路径；`-1反转` 和 `+1反转` 完全不参与。`start_date` 和 `end_date` 都是实际执行日，`holding_days` 是连续交易日数量。

In [4]:
zero_segments = pd.read_csv(OUTPUT_DIR / '持有段明细_仅零段反转.csv', encoding='utf-8-sig', parse_dates=['start_date', 'end_date'])
zero_duration = pd.read_csv(OUTPUT_DIR / '持有段天数分布_仅零段反转.csv', encoding='utf-8-sig')
print('场景：基础三状态 + 0→-1 / 0→+1 持有路径；不加入 -1→0 / +1→0')
display(zero_duration.pivot(index='holding_days', columns='state_label', values='segments').fillna(0).astype(int))
display(zero_segments[['run_id', 'state', 'start_date', 'end_date', 'holding_days', 'source_detail', 'zero_transfer_days', 'segment_return', 'return_available']])
print('以上明细覆盖每一个连续段；日期均为实际执行日，不是形成日。')

场景：基础三状态 + 0→-1 / 0→+1 持有路径；不加入 -1→0 / +1→0


state_label,-1,0,1
holding_days,,,
1,13,42,11
2,58,36,14
3,16,33,14
4,3,19,23
5,9,17,13
6,14,12,3
7,7,11,4
8,4,13,5
9,2,6,2


,run_id,state,start_date,end_date,holding_days,source_detail,zero_transfer_days,segment_return,return_available
0,1,0,2018-01-03,2018-01-15,9,基础0,0,0.000000,True
1,2,-1,2018-01-16,2018-01-23,6,0转-1持有；基础-1,1,-0.017303,True
2,3,0,2018-01-24,2018-02-01,7,基础0,0,0.000000,True
3,4,-1,2018-02-02,2018-02-05,2,0转-1持有,2,0.008647,True
4,5,0,2018-02-06,2018-02-08,3,基础0,0,0.000000,True
...,...,...,...,...,...,...,...,...,...
443,444,-1,2026-07-27,2026-08-03,6,基础-1,0,0.005713,True
444,445,0,2026-08-04,2026-08-10,5,基础0,0,0.000000,True
445,446,-1,2026-08-11,2026-08-18,6,0转-1持有,6,-0.006872,True
446,447,0,2026-08-19,2026-08-21,3,基础0,0,0.000000,True


以上明细覆盖每一个连续段；日期均为实际执行日，不是形成日。


## 5. 四个反转全部加入后的碎片诊断

这里使用最终 `Adj 1545`：四个反转全部参与。重点查看三状态的完整持有期分布，以及一日/两日段的日期、形成日和形成机制。该诊断不自动把短段合并，因为合并会改变冻结状态定义。

In [5]:
final_duration = pd.read_csv(OUTPUT_DIR / '最终三状态_持有段天数分布.csv', encoding='utf-8-sig')
short_detail = pd.read_csv(OUTPUT_DIR / '最终三状态_一二日段明细.csv', encoding='utf-8-sig', parse_dates=['start_date', 'end_date', 'start_formation_date', 'end_formation_date'])
mechanism = pd.read_csv(OUTPUT_DIR / '最终三状态_一二日段形成机制统计.csv', encoding='utf-8-sig')
print('最终四反转 Adj 1545 的持有天数分布：')
display(final_duration.pivot(index='holding_days', columns='state_label', values='segments').fillna(0).astype(int))
print('一日/两日段形成机制：')
display(mechanism)
print('一日/两日段日期明细：')
display(short_detail)

最终四反转 Adj 1545 的持有天数分布：


state_label,-1,0,1
holding_days,,,
1,13,44,16
2,58,39,17
3,16,36,14
4,6,27,26
5,20,17,16
6,10,12,5
7,5,10,7
8,2,16,5
9,1,7,2


一日/两日段形成机制：


,holding_days,state,state_label,formation_mechanism,segments
0,1,-1,-1,0转-1持有路径短段,13
1,1,0,0,基础0短段,21
2,1,0,0,0转-1与0转+1同日冲突置0,16
3,1,0,0,退出反转切出的0段,7
4,1,1,1,0转+1持有路径短段,11
5,1,1,1,基础+1短段,5
6,2,-1,-1,0转-1持有路径短段,58
7,2,0,0,基础0短段,18
8,2,0,0,0转-1与0转+1同日冲突置0,16
9,2,0,0,退出反转切出的0段,5


一日/两日段日期明细：


,series,run_id,state,state_label,start_date,end_date,start_formation_date,end_formation_date,phase,holding_days,...,formation_mechanism,source_detail,segment_return,return_available,+1反转_days,-1反转_days,0转-1_days,0转+1_days,大涨_days,大跌_days
0,Adj 1545,4,-1,-1,2018-02-02,2018-02-05,2018-02-01,2018-02-02,Train,2,...,0转-1持有路径短段,0转-1持有,0.008647,True,0,0,2,0,1,0
1,Adj 1545,9,-1,-1,2018-03-06,2018-03-07,2018-03-05,2018-03-06,Train,2,...,0转-1持有路径短段,0转-1持有,0.000517,True,0,0,2,0,0,0
2,Adj 1545,11,0,0,2018-03-14,2018-03-14,2018-03-13,2018-03-13,Train,1,...,退出反转切出的0段,+1退出到0,0.000000,True,1,0,0,0,0,0
3,Adj 1545,12,1,1,2018-03-15,2018-03-16,2018-03-14,2018-03-15,Train,2,...,基础+1短段,基础+1,-0.004905,True,0,0,0,0,0,0
4,Adj 1545,16,-1,-1,2018-03-30,2018-04-02,2018-03-29,2018-03-30,Train,2,...,0转-1持有路径短段,0转-1持有,0.001987,True,0,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,Adj 1545,466,0,0,2026-05-14,2026-05-15,2026-05-13,2026-05-14,Test,2,...,退出反转切出的0段,+1退出到0,0.000000,True,2,0,0,0,1,0
183,Adj 1545,467,1,1,2026-05-18,2026-05-18,2026-05-15,2026-05-15,Test,1,...,基础+1短段,基础+1,0.003666,True,0,0,0,0,1,1
184,Adj 1545,469,0,0,2026-05-25,2026-05-25,2026-05-22,2026-05-22,Test,1,...,0转-1与0转+1同日冲突置0,零段同日冲突置0,0.000000,True,0,0,1,1,0,1
185,Adj 1545,472,0,0,2026-06-04,2026-06-05,2026-06-03,2026-06-04,Test,2,...,基础0短段,基础0,0.000000,True,0,0,0,0,0,0
